# Stage 4: Machine-Learning Surrogate Model Development
## Project: AI-Enabled Digital Twin for Fouling-Aware Optimization of Textile Wastewater Reuse

> **Scientific Context:** This notebook investigates machine-learning surrogate models (Linear Regression, Random Forest, XGBoost, and Artificial Neural Networks) trained on simulation data from the validated mechanistic reverse-osmosis (RO) model. The primary goal is to evaluate whether nonlinear ML surrogates accurately capture complex multi-stage membrane behavior within the engineering-screened operational domain, remain robust near boundaries, and provide transparent interpretability via SHAP.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Setup paths and plot styles
sys.path.insert(0, str(Path("..").resolve() / "src"))

from ml.preprocessing import PreprocessingPipeline, FEATURE_COLUMNS, ALL_TARGETS, PRIMARY_TARGETS, SECONDARY_TARGETS
from ml.metrics import calculate_regression_metrics
from ml.evaluate import predict_model_targets, evaluate_model_on_dataset, evaluate_baseline_point
from ml.inference import Stage4Surrogate, benchmark_surrogate_speed
from ml.physics_checks import BASELINE_POINT, evaluate_mass_balance_reconstruction

plt.style.use("default")
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

## 1. Load Curated Datasets and Model Metadata

In [ ]:
data_dir = Path("../data/generated")
models_dir = Path("../models/stage4")
results_tab_dir = Path("../results/stage4/tables")

df_curated = pd.read_csv(data_dir / "stage3_engineering_acceptable.csv")
df_boundary = pd.read_csv(data_dir / "stage3_boundary_stress.csv")
df_ood = pd.read_csv(data_dir / "stage3_ood_scenarios.csv")

df_train = df_curated[df_curated["dataset_split"] == "train"].copy().reset_index(drop=True)
df_val = df_curated[df_curated["dataset_split"].isin(["val", "validation"])].copy().reset_index(drop=True)
df_test = df_curated[df_curated["dataset_split"] == "test"].copy().reset_index(drop=True)

print(f"Curated Engineering-Acceptable Scenarios : {len(df_curated):,d}")
print(f"  - Train (70%)                          : {len(df_train):,d}")
print(f"  - Validation (15%)                     : {len(df_val):,d}")
print(f"  - Primary Test Set (15%)               : {len(df_test):,d}")
print(f"Boundary Stress Dataset (>30% elem rec)  : {len(df_boundary):,d}")
print(f"Synthetic OOD Stress Dataset             : {len(df_ood):,d}")

## 2. Primary Test-Set Performance Ledger (Model Comparison)

In [ ]:
df_test_metrics = pd.read_csv(results_tab_dir / "test_metrics.csv")
display_cols = ["Model", "Target", "R2", "RMSE", "MAE", "NRMSE_pct", "MAPE"]
df_test_metrics[display_cols].sort_values(by=["Target", "R2"], ascending=[True, False])

## 3. Boundary-Stress & Synthetic OOD Generalization

In [ ]:
df_boundary_metrics = pd.read_csv(results_tab_dir / "boundary_metrics.csv")
df_ood_metrics = pd.read_csv(results_tab_dir / "ood_metrics.csv")

print("=== BOUNDARY STRESS PERFORMANCE (XGBoost vs Linear) ===")
print(df_boundary_metrics[df_boundary_metrics["Model"].isin(["XGBoost", "Linear Regression")][display_cols].to_string(index=False))

print("\n=== SYNTHETIC OOD GENERALIZATION (XGBoost) ===")
print(df_ood_metrics[df_ood_metrics["Model"] == "XGBoost"][display_cols].to_string(index=False))

## 4. Stage 2 Baseline Operating Point Verification

In [ ]:
df_base_eval = pd.read_csv(results_tab_dir / "baseline_point_metrics.csv")
df_base_eval[df_base_eval["Model"] == "XGBoost"]

## 5. Reconstructed Mass & Solute Balance Quality

In [ ]:
df_mass = pd.read_csv(results_tab_dir / "mass_balance_reconstruction.csv")
print(f"Solute Balance Error (%): Mean={df_mass['solute_error_percent'].mean():.3f}%, Median={df_mass['solute_error_percent'].median():.3f}%, 95th={df_mass['solute_error_percent'].quantile(0.95):.3f}%")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_mass["solute_error_percent"], bins=30, color="#3498db", edgecolor="black", alpha=0.8)
ax.set_xlabel("Reconstructed Solute Balance Discrepancy (%)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Solute Balance Errors (Test Set)")
ax.grid(True, linestyle=":", alpha=0.6)
plt.show()

## 6. Inference Speedup Benchmark

In [ ]:
with open(results_tab_dir / "speed_benchmark.json", "r") as f:
    speed_data = json.load(f)

print(f"Mechanistic Solver Speed : {speed_data['mechanistic_evals_per_sec']:.2f} evals/sec ({speed_data['mechanistic_time_per_eval_ms']:.2f} ms/eval)")
print(f"XGBoost Surrogate Speed  : {speed_data['surrogate_evals_per_sec']:.2f} evals/sec ({speed_data['surrogate_ms_per_eval']:.4f} ms/eval)")
print(f"Speed-Up Factor          : {speed_data['speedup_factor']:.1f}x")